In [1]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta


In [5]:
#使用平台获取行情
def stock_daily(code, start_date, end_date):
    df = ak.stock_zh_a_hist(code, period="daily", start_date=start_date, end_date=end_date, adjust="")
    df['日期'] = pd.to_datetime(df['日期'])
    df.set_index('日期', inplace=True)
    df.sort_index(inplace=True)
    return df


In [13]:
#整合买卖信号
def  allsingal(df):
    df['buy_signal']=np.where((df['buy_signal']==1) & (df['buy_signal'].shift(1) ==1) ,0,df['buy_signal'])#只保留第一个信号
    df['sell_signal']=np.where((df['sell_signal']==-1) & (df['sell_signal'].shift(1) ==-1) ,0,df['sell_signal'])
    df['signal']=df['buy_signal']+df['sell_signal']
    df.drop([ 'buy_signal','sell_signal'],axis=1) 
    df=df[df['signal'] !=0 ]         
    return df              #返回时间序列变成了仅交易日序列，整合买卖信号的df


In [15]:
#按照每笔交易计算收益率(而非每日净值)
def  cal_r(df):
    df['r']=df['收盘'].pct_change()  #收益率:当天交易收盘价-上一次交易收盘价/当天交易收盘价，不是涨跌幅，时间序列是跳跃的
    df=df[df['signal'] ==-1 ]          #时间序列变成了仅卖出日序列
    df['cumr']=(1+df['r']).cumprod() - 1   #按照每笔交易计算累计收益率
    return df           #返回仅有卖出日时间序列，增加了卖出日收益率、累计收益率的df


In [19]:
#双均线策略
def ma_strategy(df, short_window, long_window):
    #计算长短均线
    df['short_ma']=df['涨跌幅'].rolling(short_window).mean()
    df['long_ma']=df['涨跌幅'].rolling(long_window).mean()
    #生成买卖信号
    df['buy_signal']=np.where((df['short_ma']>df['long_ma']),1,0)
    df['sell_signal']=np.where((df['short_ma']<df['long_ma']),-1,0)
    #整合信号
    df=allsingal(df)  
    return df[['收盘','short_ma','long_ma','signal']]    


In [114]:
codes=["002594","300750","601012"]
end_date=datetime.now().strftime("%Y%m%d")
start_date = (datetime.now()- timedelta(days=365)).strftime("%Y%m%d")
df=stock_daily("002594", start_date=start_date, end_date=end_date) 
from scipy.stats import ttest_1samp


In [120]:
#寻找最优长短均线组合
def all_ma(df):
    parms=[5,10,20,60,120]
    all_ma=pd.DataFrame(columns=['short','long','cumr','sharpe','p']) 
    for short in parms:                     
        for long in parms:
            if long>short:
                df_ma=ma_strategy(df,short_window=short, long_window=long)
                df_ma_r=cal_r(df_ma)
                cumr=df_ma_r['cumr'].iloc[-1]                                   #输出累计收益率              
                sharpe=df_ma_r['r'].mean() / df_ma_r['r'].std() *np.sqrt(242)      #输出年夏普比
                t,p=ttest_1samp(df_ma_r['r'],0,nan_policy="omit")          #检验收益是否显著大于0
                p_value=p/2                                         
                new=pd.DataFrame([[short,long,cumr,sharpe,p_value]],columns=['short','long','cumr','sharpe','p'])
                all_ma=pd.concat([all_ma,new],ignore_index=True)                  #不同组合集合对比
    all_ma=all_ma.sort_values(by='sharpe',ascending=False)         #按收益率或者夏普倒序排列
    print(all_ma)

In [126]:
for code in codes:
    df=stock_daily(code, start_date=start_date, end_date=end_date) 
    all_ma(df)


C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\1734533222.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cumr']=(1+df['r']).cumprod() - 1   #按照每笔交易计算累计收益率
C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\643236531.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ma=pd.concat([all_ma,new],ignore_index=True)                  #不同组合集合对比
C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\1734533222.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

  short long      cumr    sharpe         p
7    20   60  0.373479  8.798629  0.137320
5    10   60  0.551964  6.109056  0.110979
4    10   20  0.746992  5.697021  0.081800
6    10  120  0.250795  4.353641  0.227296
2     5   60  0.237661  3.780640  0.173205
1     5   20  0.208054  3.160132  0.170267
8    20  120  0.035423  1.976795  0.395201
0     5   10  0.130164  1.862578  0.258563
3     5  120 -0.009276  0.125352  0.489116
9    60  120 -0.067256       NaN       NaN


C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\1734533222.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cumr']=(1+df['r']).cumprod() - 1   #按照每笔交易计算累计收益率
C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\643236531.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ma=pd.concat([all_ma,new],ignore_index=True)                  #不同组合集合对比
C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\1734533222.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

  short long      cumr     sharpe         p
1     5   20  0.246079   3.017479  0.198256
3     5  120  0.033000   2.494912  0.331915
0     5   10  0.195005   2.163115  0.226221
7    20   60  0.116280   2.100207  0.339732
2     5   60  0.138814   1.981653  0.287783
5    10   60  0.093732   1.604918  0.352863
4    10   20  0.080426   1.337295  0.352459
8    20  120 -0.014860  -1.743441  0.397333
9    60  120 -0.078674  -5.246251  0.358345
6    10  120 -0.164901 -16.055002  0.004891
  short long      cumr    sharpe         p
7    20   60  0.450689  7.394891  0.173861
5    10   60  0.248803  3.680513  0.214928
6    10  120  0.054110  3.394026  0.253817
2     5   60  0.235520  2.543512  0.215665
8    20  120  0.015579  2.265923  0.435335
1     5   20 -0.040503 -0.142559  0.483061
0     5   10 -0.097671 -0.693768  0.412717
4    10   20 -0.083520 -1.066203  0.375515
3     5  120 -0.058157 -1.987285  0.333340
9    60  120 -0.160603       NaN       NaN


C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\1734533222.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cumr']=(1+df['r']).cumprod() - 1   #按照每笔交易计算累计收益率
C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\643236531.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ma=pd.concat([all_ma,new],ignore_index=True)                  #不同组合集合对比
C:\Users\ryo\AppData\Local\Temp\ipykernel_20600\1734533222.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram